In [5]:
import yfinance as yf
import pandas as pd

def get_brent(days: int = 150) -> pd.DataFrame:
    df = yf.download("BZ=F", period=f"{days}d", interval="1d", progress=False)
    
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    
    df = df.reset_index()
    df = df[["Date", "Close"]]
    df["Date"] = pd.to_datetime(df["Date"]).dt.tz_localize(None)
    df = df.dropna()
    df = df.sort_values("Date").reset_index(drop=True)
    
    return df

df_brent = get_brent()
print(df_brent.shape)
print(df_brent.head(3))
print(df_brent.dtypes)


1 Failed download:
['BZ=F']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')


(0, 2)
Empty DataFrame
Columns: [Date, Close]
Index: []
Price
Date     datetime64[ns]
Close           float64
dtype: object


In [7]:
import requests
import pandas as pd

url = "https://iss.moex.com/iss/engines/futures/markets/forts/securities.json?q=BR"
resp = requests.get(url)
data = resp.json()

columns = data["securities"]["columns"]
rows    = data["securities"]["data"]

df = pd.DataFrame(rows, columns=columns)
print(df.columns.tolist())
print(df[df["SECID"].str.startswith("BR")][["SECID", "SHORTNAME"]].head(20))

['SECID', 'BOARDID', 'SHORTNAME', 'SECNAME', 'PREVSETTLEPRICE', 'DECIMALS', 'MINSTEP', 'LASTTRADEDATE', 'LASTDELDATE', 'SECTYPE', 'LATNAME', 'ASSETCODE', 'PREVOPENPOSITION', 'LOTVOLUME', 'INITIALMARGIN', 'HIGHLIMIT', 'LOWLIMIT', 'STEPPRICE', 'LASTSETTLEPRICE', 'PREVPRICE', 'IMTIME', 'BUYSELLFEE', 'SCALPERFEE', 'NEGOTIATEDFEE', 'EXERCISEFEE', 'SETTLEPRICE_CLR']
   SECID SHORTNAME
33  BRK6   BR-5.26
34  BRM6   BR-6.26
35  BRN6   BR-7.26
36  BRQ6   BR-8.26
37  BRU6   BR-9.26
38  BRV6  BR-10.26
39  BRX6  BR-11.26


In [8]:
import requests
import pandas as pd
from datetime import datetime, timedelta

date_from = (datetime.now() - timedelta(days=150)).strftime("%Y-%m-%d")
date_to   = datetime.now().strftime("%Y-%m-%d")

url = (
    f"https://iss.moex.com/iss/engines/futures/markets/forts"
    f"/boards/RFUD/securities/BRK6/candles.json"
    f"?from={date_from}&till={date_to}&interval=24"
)

resp = requests.get(url)
data = resp.json()

columns = data["candles"]["columns"]
rows    = data["candles"]["data"]

df = pd.DataFrame(rows, columns=columns)
print(f"Строк: {len(df)}")
print(df.head(5))

Строк: 101
    open  close   high    low  value  volume                begin  \
0  63.22  63.40  63.40  62.73      0      23  2025-11-17 00:00:00   
1  63.28  62.48  63.70  62.05      0      54  2025-11-19 00:00:00   
2  62.41  62.80  62.80  62.07      0      25  2025-11-20 00:00:00   
3  62.64  61.13  62.69  60.73      0     321  2025-11-21 00:00:00   
4  61.50  61.33  61.88  61.00      0      79  2025-11-24 00:00:00   

                   end  
0  2025-11-17 23:59:59  
1  2025-11-19 23:59:59  
2  2025-11-20 23:59:59  
3  2025-11-21 23:59:59  
4  2025-11-24 23:59:59  


In [9]:
def get_nearest_brent_ticker() -> str:
    url  = "https://iss.moex.com/iss/engines/futures/markets/forts/securities.json?q=BR"
    resp = requests.get(url, timeout=10)
    data = resp.json()

    columns = data["securities"]["columns"]
    rows    = data["securities"]["data"]

    df = pd.DataFrame(rows, columns=columns)
    df = df[df["ASSETCODE"] == "BR"][["SECID", "LASTTRADEDATE"]].copy()
    df["LASTTRADEDATE"] = pd.to_datetime(df["LASTTRADEDATE"])
    df = df[df["LASTTRADEDATE"] >= pd.Timestamp.now()]
    df = df.sort_values("LASTTRADEDATE")

    return df.iloc[0]["SECID"]

ticker = get_nearest_brent_ticker()
print(f"Ближайший контракт: {ticker}")

Ближайший контракт: BRK6
